# GRPO and reward hacking from scratch (numpy only)

Four experiments, each matching a section of `notes.md`:

1. **Zero-signal groups** (notes §4): the fraction of GRPO groups whose
   rewards are all equal matches $p^G + (1-p)^G$.
2. **GRPO with a strict verifier** (notes §1, §5): reward and accuracy rise
   together, while entropy falls and more groups stop giving signal.
3. **GRPO with a loose verifier** (notes §2, §3): the training reward
   reaches 0.99, but accuracy ends far below the strict run, because the
   policy learns to list every number on the prompts it can't solve.
4. **Best-of-N KL** (notes §9): a simulation matches $\log N - (N-1)/N$.

The toy task is one-digit addition. The policy is a table: for each of the
100 prompts $a + b$ it has a softmax over 20 actions, the answers 0 to 18
plus one "spam" action that outputs every number from 0 to 18. A real model
generates many tokens; here a completion is one action, which keeps every
step visible while the GRPO update stays the same.

In [1]:
import numpy as np
rng = np.random.default_rng(0)

PROMPTS = [(a, b) for a in range(10) for b in range(10)]
ANSWERS = np.array([a + b for a, b in PROMPTS])
P, V = len(PROMPTS), 20
SPAM = 19

def text(action):
    """The completion an action writes."""
    return " ".join(str(k) for k in range(19)) if action == SPAM else str(action)

def strict_reward(i, action):
    """Parse the completion as one number and compare it to the answer."""
    return float(text(action) == str(ANSWERS[i]))

def loose_reward(i, action):
    """The buggy checker from exercise 7: is the answer anywhere in the text?"""
    return float(str(ANSWERS[i]) in text(action))

def softmax(z):
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)

The starting policy plays the role of the SFT model. Each prompt's correct
answer gets a random boost, so pass rates vary from prompt to prompt the
way they do in a real prompt set: some prompts are nearly always solved,
some almost never. The spam action starts at about 3% probability.

In [2]:
ref_logits = rng.normal(0, 1, (P, V))
ref_logits[np.arange(P), ANSWERS] += rng.uniform(-1, 5, P)
ref_logits[:, SPAM] = 0.0
pi_ref = softmax(ref_logits)
pass_rate = pi_ref[np.arange(P), ANSWERS]

print("pass rate quantiles (0, 25, 50, 75, 100%):", np.round(np.quantile(pass_rate, [0, .25, .5, .75, 1]), 3))
print("mean pass rate:", round(pass_rate.mean(), 3))
print("mean probability of spam:", round(pi_ref[:, SPAM].mean(), 3))

pass rate quantiles (0, 25, 50, 75, 100%): [0.003 0.042 0.133 0.393 0.911]
mean pass rate: 0.251
mean probability of spam: 0.027


## 1. Zero-signal groups

With $G = 8$ samples per prompt and a 0/1 reward, a prompt with pass rate
$p$ gives a group of identical rewards with probability $p^8 + (1-p)^8$.
Averaged over the prompt set, that predicts how many groups will give no
gradient. Sample many groups and count.

In [3]:
G = 8
predicted = np.mean(pass_rate**G + (1 - pass_rate)**G)

trials, same = 200, 0
for _ in range(trials):
    for i in range(P):
        acts = rng.choice(V, size=G, p=pi_ref[i])
        r = [strict_reward(i, a) for a in acts]
        same += (min(r) == max(r))
print(f"predicted zero-signal fraction: {predicted:.3f}")
print(f"simulated zero-signal fraction: {same / (trials * P):.3f}")

predicted zero-signal fraction: 0.405
simulated zero-signal fraction: 0.402


About 40% of the groups give nothing, before any training. This is the
compute that pass-rate filtering and dynamic sampling (notes §4) recover.

## 2. GRPO with a strict verifier

Each step: sample 32 prompts, draw $G = 8$ actions per prompt from the
current policy, score them, and compute GRPO advantages
$(r_i - \text{mean})/\text{std}$ within each group. Then take two gradient
steps on the clipped objective. For a softmax policy, the gradient of
$\log \pi(a)$ with respect to the logits is $\text{onehot}(a) - \pi$, so
an unclipped sample contributes $A \rho\,(\text{onehot}(a) - \pi)$.

The optional KL term uses the $k_3$ estimator from notes §5,
$k_3 = (u - 1) - \log u$ with $u = \pi_{ref}(a)/\pi(a)$. Its gradient
with respect to the logits is $(1 - u)(\text{onehot}(a) - \pi)$, and we
step against it.

`grpo` logs, after every step: the expected reward under the checker being
trained on, the expected strict accuracy, the probability of spam,
the entropy, the KL to the starting policy, and the fraction of groups
with zero advantage.

In [4]:
def expected_reward(pi, reward_fn):
    return np.mean([sum(pi[i, a] * reward_fn(i, a) for a in range(V)) for i in range(P)])

def grpo(reward_fn, steps=300, B=32, G=8, lr=20.0, eps_lo=0.2, eps_hi=0.2,
         beta=0.0, epochs=2, seed=1):
    rng = np.random.default_rng(seed)
    logits = ref_logits.copy()
    log = []
    for step in range(steps):
        pi = softmax(logits)
        idx = rng.choice(P, B, replace=False)
        acts = np.array([rng.choice(V, size=G, p=pi[i]) for i in idx])
        R = np.array([[reward_fn(i, a) for a in row] for i, row in zip(idx, acts)])
        mu, sd = R.mean(1, keepdims=True), R.std(1, keepdims=True)
        A = np.where(sd > 0, (R - mu) / np.maximum(sd, 1e-8), 0.0)
        zero_frac = (sd[:, 0] == 0).mean()
        p_old = pi[idx[:, None], acts]          # probabilities under the sampling policy

        for _ in range(epochs):
            pi_now = softmax(logits)
            grad = np.zeros_like(logits)
            for bi, i in enumerate(idx):
                for g in range(G):
                    a, adv = acts[bi, g], A[bi, g]
                    rho = pi_now[i, a] / p_old[bi, g]
                    d_logp = -pi_now[i].copy(); d_logp[a] += 1   # onehot(a) - pi
                    clipped = (adv > 0 and rho > 1 + eps_hi) or (adv < 0 and rho < 1 - eps_lo)
                    if not clipped:
                        grad[i] += adv * rho * d_logp
                    if beta:
                        u = pi_ref[i, a] / pi_now[i, a]
                        grad[i] -= beta * (1 - u) * d_logp
            logits += lr * grad / (B * G)

        pi = softmax(logits)
        log.append(dict(
            step=step,
            train_reward=expected_reward(pi, reward_fn),
            accuracy=pi[np.arange(P), ANSWERS].mean(),
            spam=pi[:, SPAM].mean(),
            entropy=-(pi * np.log(pi + 1e-12)).sum(1).mean(),
            kl=(pi * (np.log(pi + 1e-12) - np.log(pi_ref))).sum(1).mean(),
            zero_frac=zero_frac,
        ))
    return log, softmax(logits)

def show(log, every=50):
    keys = ["train_reward", "accuracy", "spam", "entropy", "kl", "zero_frac"]
    print("step  " + "  ".join(f"{k:>12}" for k in keys))
    for row in log[::every] + [log[-1]]:
        print(f"{row['step']:>4}  " + "  ".join(f"{row[k]:12.3f}" for k in keys))

strict_log, strict_pi = grpo(strict_reward)
show(strict_log)

step  train_reward      accuracy          spam       entropy            kl     zero_frac
   0         0.263         0.263         0.027         2.258         0.003         0.438
  50         0.671         0.671         0.013         1.094         0.736         0.781
 100         0.829         0.829         0.006         0.590         1.310         0.906
 150         0.879         0.879         0.005         0.417         1.531         0.875
 200         0.903         0.903         0.004         0.332         1.642         0.906
 250         0.934         0.934         0.003         0.245         1.782         0.906
 299         0.952         0.952         0.002         0.175         1.881         0.938


Reward and accuracy are the same number here, because the checker is
correct. Accuracy climbs from 0.26 to about 0.95. Two other columns show
the patterns from notes §7: entropy falls as the policy concentrates on
the answers it has found, and the zero-signal fraction climbs, because
most prompts become solved by all 8 samples. Late in training, most of the
generation compute produces no gradient.

## 3. The same run with the loose verifier

Now train against `loose_reward`, which gives 1 whenever the answer string
appears anywhere in the completion. The spam action always gets 1. So do
some wrong answers: for the prompt $5 + 6 = 11$, the answer "1" is not
rewarded, but for $0 + 1 = 1$ the answer "11" is, since "1" appears in it.

In [5]:
loose_log, loose_pi = grpo(loose_reward)
show(loose_log)

print()
print(f"strict run: final accuracy {strict_log[-1]['accuracy']:.3f}")
print(f"loose run:  final train reward {loose_log[-1]['train_reward']:.3f}, final accuracy {loose_log[-1]['accuracy']:.3f}")

step  train_reward      accuracy          spam       entropy            kl     zero_frac
   0         0.317         0.264         0.027         2.257         0.003         0.312
  50         0.788         0.611         0.083         0.937         0.923         0.656
 100         0.951         0.671         0.165         0.323         1.590         0.875
 150         0.976         0.681         0.178         0.191         1.723         0.781
 200         0.988         0.692         0.178         0.122         1.795         0.906
 250         0.991         0.694         0.179         0.093         1.820         0.969
 299         0.993         0.696         0.179         0.077         1.835         0.906

strict run: final accuracy 0.952
loose run:  final train reward 0.993, final accuracy 0.696


The loose run reports a training reward of 0.99, higher than the strict
run's, while its real accuracy ends much lower. This is the
over-optimization picture from notes §3: the number being optimized keeps
going up, and the number you care about doesn't.

Where did the policy spam? Split the prompts by their starting pass rate.

In [6]:
spam_prob = loose_pi[:, SPAM]
for lo, hi in [(0, 0.05), (0.05, 0.3), (0.3, 1.01)]:
    m = (pass_rate >= lo) & (pass_rate < hi)
    print(f"starting pass rate in [{lo:.2f}, {hi:.2f}): {m.sum():3d} prompts, "
          f"mean final spam probability {spam_prob[m].mean():.3f}, "
          f"accuracy {loose_pi[m][:, ANSWERS[m]].diagonal().mean():.3f}")

starting pass rate in [0.00, 0.05):  30 prompts, mean final spam probability 0.527, accuracy 0.201
starting pass rate in [0.05, 0.30):  35 prompts, mean final spam probability 0.058, accuracy 0.823
starting pass rate in [0.30, 1.01):  35 prompts, mean final spam probability 0.000, accuracy 0.993


The policy spams mostly where the correct answer was hard to find. There,
spam is the easiest way to reward 1, and once spam earns 1 every time,
a correct answer earns no more than spam does, so nothing pushes the
policy toward the correct one. On prompts it already solved, it keeps
answering correctly. Real hacks look like this: concentrated on the hard
cases, and invisible in the average training reward.

**Does a KL penalty fix it?** Rerun the loose verifier with three values
of $\beta$.

In [7]:
print(" beta  train_reward  accuracy   spam     kl")
for beta in [0.0, 0.1, 1.0]:
    log, _ = grpo(loose_reward, beta=beta)
    r = log[-1]
    print(f"{beta:5.1f}  {r['train_reward']:12.3f}  {r['accuracy']:8.3f}  {r['spam']:.3f}  {r['kl']:.3f}")

 beta  train_reward  accuracy   spam     kl


  0.0         0.993     0.696  0.179  1.835


  0.1         0.969     0.674  0.181  1.646


  1.0         0.683     0.484  0.131  0.473


A larger $\beta$ keeps the policy closer to where it started, which slows
the spam, but it slows the correct learning just as much: accuracy falls
with it. KL limits how far the policy moves; it cannot tell a hack from
progress. The fix for a broken checker is fixing the checker, which is the
strict run in §2.

## 4. The KL of best-of-N

Take a policy over 1,000 completions with random probabilities and a
random reward for each, so there are no ties. Best-of-$N$ draws $N$
completions and keeps the highest-reward one. Estimate its output
distribution by simulation, compute its KL to the policy, and compare with
$\log N - (N-1)/N$ (exercise 5).

In [8]:
K = 1000
pi = rng.dirichlet(np.ones(K))
reward = rng.normal(size=K)

for N in [4, 16, 64]:
    draws = rng.choice(K, size=(100_000, N), p=pi)
    best = draws[np.arange(len(draws)), reward[draws].argmax(1)]
    pi_N = np.bincount(best, minlength=K) / len(best)
    m = pi_N > 0
    kl = np.sum(pi_N[m] * np.log(pi_N[m] / pi[m]))
    print(f"N = {N:3d}: simulated KL {kl:.3f}, formula {np.log(N) - (N - 1) / N:.3f}")

N =   4: simulated KL 0.643, formula 0.636
N =  16: simulated KL 1.830, formula 1.835


N =  64: simulated KL 3.175, formula 3.175


The simulation matches the formula up to sampling noise, which grows with
$N$ because best-of-64 concentrates on fewer completions. The KL grows
like $\log N$, so best-of-$N$ moves surprisingly little from the policy
even at large $N$. That makes it a cheap reference point for an RL run at
the same KL.